# **BindingDB Fine-tuning**

## Environment and Input Verification

This section checks the Python environment, GPU, package versions, and required project files. It does not load the model, modify files, or start training.

In [1]:
import os
import sys
import platform
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version

import torch
from rdkit import rdBase

In [2]:
REPO_ROOT = Path(
    os.environ.get("CTDDG_REPO_ROOT", "/home/muhamed/repo/CTDDG")
).expanduser().resolve()

VARIANT = "ctddg_eq8"  # Alternative: "cdgcn_nll"

EXPERIMENT_CLASS = "exploratory_leaked_pretraining"

RUN_TRAINING = False
EVALUATE_TEST = False
RUN_DIR = None  # Choose only after provenance verification.

DATA_DIR = (
    REPO_ROOT / "datasets/processed/ctddg_bindingdb_rf_d1_v2"
)

ATOM_TYPES_FILE = (
    REPO_ROOT / "REWORKED_CODE/chembl_preprocessed/atom_types.txt"
)

PRETRAIN_RUNS = {
    "ctddg_eq8": (
        REPO_ROOT
        / "REWORKED_CODE/pretraining/pretraining_runs/ctddg_torch_eq8_bs32_v1"
    ),
    "cdgcn_nll": (
        REPO_ROOT
        / "REWORKED_CODE/pretraining/pretraining_runs/cdgcn_torch_nll_bs32_v1"
    ),
}

assert VARIANT in PRETRAIN_RUNS

In [3]:
print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("Host:", platform.node())

missing_packages = []

for package in (
    "torch", "rdkit", "numpy", "pandas",
    "scipy", "networkx", "h5py",
    "scikit-learn", "joblib"
):
    try:
        print(f"{package:14s} {version(package)}")
    except PackageNotFoundError:
        print(f"{package:14s} MISSING")
        missing_packages.append(package)

print("RDKit runtime:", rdBase.rdkitVersion)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(
        f"GPU 0: {gpu.name} | "
        f"VRAM: {gpu.total_memory / 1024**3:.1f} GiB"
    )
else:
    print(
        "GPU unavailable. File inventory is safe, "
        "but GPU preflight/training needs a Slurm allocation."
    )


Python: 3.11.16
Interpreter: /home/muhamed/.conda/envs/ctddg_torch/bin/python
Host: node004
torch          2.4.1+cu121
rdkit          2026.3.6
numpy          2.4.6
pandas         3.0.6
scipy          1.17.1
networkx       3.6.1
h5py           3.16.0
scikit-learn   1.9.1
joblib         1.6.0
RDKit runtime: 2026.03.6
PyTorch CUDA build: 12.1
CUDA available: True
GPU 0: NVIDIA L40 | VRAM: 44.4 GiB


In [4]:
required_files = {
    "BindingDB manifest": DATA_DIR / "dataset_manifest.json",
    "BindingDB train": DATA_DIR / "train.tsv",
    "BindingDB validation": DATA_DIR / "val.tsv",
    "BindingDB test (existence only)": DATA_DIR / "test.tsv",
    "Ordered atom vocabulary": ATOM_TYPES_FILE,
}

for name, directory in PRETRAIN_RUNS.items():
    required_files[f"{name} checkpoint"] = directory / "checkpoint.pt"
    required_files[f"{name} metadata"] = directory / "run_meta.json"

print("\nInput file inventory:")

missing = []

for name, path in required_files.items():
    exists = path.is_file()
    print(f"{'OK' if exists else 'MISSING':7s} {name}: {path}")
    if not exists:
        missing.append(str(path))


Input file inventory:
OK      BindingDB manifest: /home/muhamed/repo/CTDDG/datasets/processed/ctddg_bindingdb_rf_d1_v2/dataset_manifest.json
OK      BindingDB train: /home/muhamed/repo/CTDDG/datasets/processed/ctddg_bindingdb_rf_d1_v2/train.tsv
OK      BindingDB validation: /home/muhamed/repo/CTDDG/datasets/processed/ctddg_bindingdb_rf_d1_v2/val.tsv
OK      BindingDB test (existence only): /home/muhamed/repo/CTDDG/datasets/processed/ctddg_bindingdb_rf_d1_v2/test.tsv
OK      Ordered atom vocabulary: /home/muhamed/repo/CTDDG/REWORKED_CODE/chembl_preprocessed/atom_types.txt
OK      ctddg_eq8 checkpoint: /home/muhamed/repo/CTDDG/REWORKED_CODE/pretraining/pretraining_runs/ctddg_torch_eq8_bs32_v1/checkpoint.pt
OK      ctddg_eq8 metadata: /home/muhamed/repo/CTDDG/REWORKED_CODE/pretraining/pretraining_runs/ctddg_torch_eq8_bs32_v1/run_meta.json
OK      cdgcn_nll checkpoint: /home/muhamed/repo/CTDDG/REWORKED_CODE/pretraining/pretraining_runs/cdgcn_torch_nll_bs32_v1/checkpoint.pt
OK      cdgcn_n

In [5]:
print("\nExperiment:", EXPERIMENT_CLASS)
print("Selected model:", VARIANT)
print("Training enabled:", RUN_TRAINING)
print("Test evaluation enabled:", EVALUATE_TEST)

if missing_packages:
    raise ModuleNotFoundError(
        "Required packages missing: " + ", ".join(missing_packages)
    )

if missing:
    raise FileNotFoundError(
        "Missing inputs:\n" + "\n".join(missing)
    )

print("\nPASS: Required files located.")
print("NOTE: File contents, hashes and checkpoints are NOT yet verified.")


Experiment: exploratory_leaked_pretraining
Selected model: ctddg_eq8
Training enabled: False
Test evaluation enabled: False

PASS: Required files located.
NOTE: File contents, hashes and checkpoints are NOT yet verified.


## Dataset and Pretraining Verification

In [6]:
import json

manifest_path = DATA_DIR / "dataset_manifest.json"
manifest = json.loads(manifest_path.read_text())

assert manifest["schema_version"] == 2
assert manifest["fold"] == "d1"

print("BindingDB dataset")
print("-----------------")
print(f"Fold: {manifest['fold']}")
print(f"Dataset version: {manifest['schema_version']}\n")

print("Dataset sizes:")
for split in ("train", "val", "test"):
    stats = manifest["stats"][split]
    print(
        f"  {split.capitalize():10s} "
        f"{stats['pairs']:,} pairs from "
        f"{stats['unique_proteins']:,} proteins"
    )

print("\nToxicity labels:")
print(manifest["label_meaning"])

print("\nDataset information loaded successfully.")


BindingDB dataset
-----------------
Fold: d1
Dataset version: 2

Dataset sizes:
  Train      118,765 pairs from 988 proteins
  Val        6,675 pairs from 52 proteins
  Test       13,836 pairs from 104 proteins

Toxicity labels:
RF-predicted Ames positive (1) vs negative (0), not measured

Dataset information loaded successfully.


In [7]:
pretrain_metadata = {}

for variant, directory in PRETRAIN_RUNS.items():
    metadata_path = directory / "run_meta.json"
    metadata = json.loads(metadata_path.read_text())
    pretrain_metadata[variant] = metadata

    print(f"\n{'CTDDG' if variant == 'ctddg_eq8' else 'CDGCN'} pretraining")
    print("-------------------------")
    print(f"Model: {metadata.get('model')}")
    print(f"Batch size: {metadata.get('batch_size', 'Not recorded')}")
    print(f"Generation paths (K): {metadata.get('k_paths')}")
    print(f"Path randomness (alpha): {metadata.get('alpha')}")

    if variant == "ctddg_eq8":
        print(f"Toxicity weight: {metadata.get('tox_lambda')}")

    config = metadata.get("model_config", {})
    print(f"GCN layers: {len(config.get('F_h', []))}")
    print(f"GRU layers: {config.get('N_rnn', 'Not recorded')}")

    assert isinstance(config, dict) and config
    assert metadata["k_paths"] == 5
    assert abs(metadata["alpha"] - 0.8) < 1e-9

print("\nBoth pretraining records loaded successfully.")



CTDDG pretraining
-------------------------
Model: CDGCN-backbone-CTDDG-Eq8-PyTorch
Batch size: 32
Generation paths (K): 5
Path randomness (alpha): 0.8
Toxicity weight: 1.0
GCN layers: 6
GRU layers: Not recorded

CDGCN pretraining
-------------------------
Model: CDGCN-unconditional-PyTorch
Batch size: 8
Generation paths (K): 5
Path randomness (alpha): 0.8
GCN layers: 6
GRU layers: Not recorded

Both pretraining records loaded successfully.


In [8]:
import hashlib

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


vocab_hash = sha256_file(ATOM_TYPES_FILE)
bindingdb_hash = manifest["dependencies_sha256"].get(
    str(ATOM_TYPES_FILE)
)

print("Atom vocabulary")
print("---------------")

for variant, metadata in pretrain_metadata.items():
    assert metadata["atom_types_sha256"] == vocab_hash
    print(f"{variant}: Matches the saved vocabulary")

assert bindingdb_hash == vocab_hash
print("BindingDB: Matches the saved vocabulary")

print("\nPretraining data leakage")
print("------------------------")

for variant, metadata in pretrain_metadata.items():
    result = manifest["pretrain_leakage"][variant]
    status = result["status"]
    overlap = result.get("overlapping_unique_ligands")

    print(f"\n{variant}:")
    print(f"  Leakage check: {status}")

    if overlap is not None:
        print(f"  Overlapping ligands: {overlap:,}")

    if result.get("pretrain_sha256"):
        same_data = (
            metadata.get("chembl_sha256")
            == result["pretrain_sha256"]
        )
        print(f"  Pretraining data hash matches: {same_data}")
        assert same_data, f"{variant}: Pretraining data mismatch"

    if result.get("stage1_meta_sha256"):
        actual_hash = sha256_file(
            PRETRAIN_RUNS[variant] / "run_meta.json"
        )
        same_metadata = (
            actual_hash == result["stage1_meta_sha256"]
        )
        print(f"  Pretraining metadata matches: {same_metadata}")
        assert same_metadata, f"{variant}: Metadata mismatch"

    if status != "verified_zero":
        print("  Note: This checkpoint is not cleared for final results.")

print("\nVerification complete. No files were changed.")


Atom vocabulary
---------------
ctddg_eq8: Matches the saved vocabulary
cdgcn_nll: Matches the saved vocabulary
BindingDB: Matches the saved vocabulary

Pretraining data leakage
------------------------

ctddg_eq8:
  Leakage check: FAILED
  Overlapping ligands: 67,708
  Pretraining data hash matches: True
  Pretraining metadata matches: True
  Note: This checkpoint is not cleared for final results.

cdgcn_nll:
  Leakage check: FAILED
  Overlapping ligands: 67,708
  Pretraining data hash matches: True
  Pretraining metadata matches: True
  Note: This checkpoint is not cleared for final results.

Verification complete. No files were changed.


## BindingDB Dataset Verification

In [9]:
import pandas as pd

splits = {}
required_columns = [
    "smiles", "protein_index", "protein_id", "ames"
]

for name in ("train", "val"):
    path = DATA_DIR / f"{name}.tsv"

    # Verify that the dataset has not changed.
    expected_hash = manifest["output_sha256"][path.name]
    assert sha256_file(path) == expected_hash, (
        f"{name}: Dataset hash does not match."
    )

    df = pd.read_csv(path, sep="\t")

    # Check columns and missing values.
    assert set(required_columns).issubset(df.columns)
    assert not df[required_columns].isna().any().any()

    splits[name] = df

    print(f"{name.capitalize()} dataset loaded successfully.")
    print(f"  Records: {len(df):,}")
    print("  File integrity: Verified\n")

print("Both dataset files are ready for inspection.")


Train dataset loaded successfully.
  Records: 118,765
  File integrity: Verified

Val dataset loaded successfully.
  Records: 6,675
  File integrity: Verified

Both dataset files are ready for inspection.


In [10]:

for name, df in splits.items():
    expected = manifest["stats"][name]

    pairs = len(df)
    proteins = df["protein_index"].nunique()
    molecules = df["smiles"].nunique()
    toxic = int(df["ames"].sum())

    assert df["ames"].isin([0, 1]).all()
    assert (df["protein_index"] >= 0).all()

    assert pairs == expected["pairs"]
    assert proteins == expected["unique_proteins"]
    assert molecules == expected["unique_ligands"]
    assert toxic == expected["ames_positive"]

    print(f"{name.capitalize()} dataset")
    print(f"  Molecule-protein pairs: {pairs:,}")
    print(f"  Unique molecules: {molecules:,}")
    print(f"  Unique proteins: {proteins:,}")
    print(f"  Predicted toxic: {toxic / pairs:.1%}")
    print("  Statistics: Verified\n")

print("Dataset statistics match the preprocessing records.")


Train dataset
  Molecule-protein pairs: 118,765
  Unique molecules: 98,583
  Unique proteins: 988
  Predicted toxic: 33.0%
  Statistics: Verified

Val dataset
  Molecule-protein pairs: 6,675
  Unique molecules: 6,622
  Unique proteins: 52
  Predicted toxic: 40.7%
  Statistics: Verified

Dataset statistics match the preprocessing records.


In [11]:

train = splits["train"]
val = splits["val"]

for name, df in splits.items():
    expected_ids = "p" + (
        df["protein_index"].astype(int) + 1
    ).astype(str)

    assert (df["protein_id"] == expected_ids).all()

    duplicates = df.duplicated(
        ["smiles", "protein_index"]
    ).sum()

    assert duplicates == 0, (
        f"{name}: Found {duplicates} duplicate pairs."
    )

train_proteins = set(train["protein_index"])
val_proteins = set(val["protein_index"])

shared_proteins = train_proteins & val_proteins

assert not shared_proteins, (
    f"Training and validation share {len(shared_proteins)} proteins."
)

assert val_proteins == set(manifest["protein_rows_val"])

print("Protein and split verification")
print("------------------------------")
print("Protein IDs: Correct")
print("Duplicate molecule-protein pairs: None")
print("Shared training/validation proteins: None")
print("Validation protein selection: Matches preprocessing")

print("\nAll dataset separation checks passed.")


Protein and split verification
------------------------------
Protein IDs: Correct
Duplicate molecule-protein pairs: None
Shared training/validation proteins: None
Validation protein selection: Matches preprocessing

All dataset separation checks passed.


## Protein Embedding Verification

In [12]:
from pathlib import Path

protein_paths = {
    name: Path(path)
    for name, path in manifest["protein_paths"]["train"].items()
}

print("Protein embedding files")
print("-----------------------")

for name in ("list", "fasta", "h5"):
    path = protein_paths[name]

    assert path.is_file(), f"Missing file: {path}"

    expected_hash = manifest["dependencies_sha256"][str(path)]
    assert sha256_file(path) == expected_hash, (
        f"{name}: File has changed since preprocessing."
    )

    print(f"{name.upper()}: Found and verified")

print("\nAll protein files match the preprocessing records.")


Protein embedding files
-----------------------
LIST: Found and verified
FASTA: Found and verified
H5: Found and verified

All protein files match the preprocessing records.


In [13]:
# Read proteins in their original order.
protein_sequences = [
    "".join(line.split()).upper()
    for line in protein_paths["list"].read_text().splitlines()
    if line.strip()
]

# Read FASTA records without changing their order.
fasta_records = []
name, sequence = None, []

for line in protein_paths["fasta"].read_text().splitlines():
    line = line.strip()

    if line.startswith(">"):
        if name is not None:
            fasta_records.append((name, "".join(sequence).upper()))
        name = line[1:].split()[0]
        sequence = []
    elif line:
        sequence.append("".join(line.split()))

if name is not None:
    fasta_records.append((name, "".join(sequence).upper()))

assert len(protein_sequences) == len(fasta_records)
assert len(protein_sequences) == len(set(protein_sequences))

for i, (protein_id, sequence) in enumerate(fasta_records, start=1):
    assert protein_id == f"p{i}"
    assert sequence == protein_sequences[i - 1]

# Both splits refer to the original training-protein list.
used_indices = set()

for split in ("train", "val"):
    indices = set(splits[split]["protein_index"].astype(int))
    assert all(0 <= i < len(protein_sequences) for i in indices)
    used_indices.update(indices)

print("Protein sequence verification")
print("-----------------------------")
print(f"Proteins in original list: {len(protein_sequences):,}")
print(f"Proteins used in train/val: {len(used_indices):,}")
print(f"Proteins without usable pairs: {len(protein_sequences) - len(used_indices)}")
print("FASTA sequence order: Correct")
print("Protein IDs: Correct")

print("\nProtein sequence verification passed.")


Protein sequence verification
-----------------------------
Proteins in original list: 1,042
Proteins used in train/val: 1,040
Proteins without usable pairs: 2
FASTA sequence order: Correct
Protein IDs: Correct

Protein sequence verification passed.


In [14]:
import h5py
import numpy as np

N_C = 6165
n_proteins = len(protein_sequences)

protein_embeddings = np.empty(
    (n_proteins, N_C), dtype=np.float32
)

with h5py.File(protein_paths["h5"], "r") as h5:
    expected_ids = {
        f"p{i}" for i in range(1, n_proteins + 1)
    }

    assert set(h5.keys()) == expected_ids, (
        "Embedding protein IDs do not match the protein list."
    )

    for i in range(n_proteins):
        protein_id = f"p{i + 1}"
        vector = np.asarray(h5[protein_id], dtype=np.float32)

        assert vector.shape == (N_C,), (
            f"{protein_id}: Incorrect embedding size."
        )

        assert np.isfinite(vector).all(), (
            f"{protein_id}: Embedding contains NaN or infinity."
        )

        protein_embeddings[i] = vector

# Check for empty embeddings.
norms = np.linalg.norm(protein_embeddings, axis=1)

assert np.isfinite(norms).all()
assert (norms > 0).all(), "Found an all-zero protein embedding."

print("ProSE embedding verification")
print("---------------------------")
print(f"Proteins loaded: {n_proteins:,}")
print(f"Embedding size: {N_C:,} values per protein")
print("Protein IDs: All matched")
print("Invalid numerical values: None")
print("Empty embeddings: None")

print("\nAll protein embeddings loaded successfully.")


ProSE embedding verification
---------------------------
Proteins loaded: 1,042
Embedding size: 6,165 values per protein
Protein IDs: All matched
Invalid numerical values: None
Empty embeddings: None

All protein embeddings loaded successfully.


## Model Architecture and Pretrained Weights

In [15]:
config = pretrain_metadata[VARIANT]["model_config"]

# Verify the architecture.
atom_count = sum(
    bool(line.strip())
    for line in ATOM_TYPES_FILE.read_text().splitlines()
)

assert config["n_atom"] == atom_count
assert config["n_bond"] == 4
assert config["D"] == 2
assert len(config["F_h"]) == 6
assert config["n_rnn"] == 3

print("Pretrained model architecture")
print("-----------------------------")
print(f"Model: {VARIANT}")
print(f"Atom types: {config['n_atom']}")
print(f"Bond types: {config['n_bond']}")
print(f"GCN layers: {len(config['F_h'])}")
print(f"GCN layer sizes: {config['F_h']}")
print(f"GRU layers: {config['n_rnn']}")
print(f"GRU hidden size: {config['F_c'][-1]}")
print(f"Atom embedding size: {config['F_e']}")
print(f"Graph receptive field: {config['D']}")

print("\nArchitecture configuration verified.")


Pretrained model architecture
-----------------------------
Model: ctddg_eq8
Atom types: 43
Bond types: 4
GCN layers: 6
GCN layer sizes: [32, 64, 128, 128, 256, 256]
GRU layers: 3
GRU hidden size: 512
Atom embedding size: 16
Graph receptive field: 2

Architecture configuration verified.


In [16]:

checkpoint_path = PRETRAIN_RUNS[VARIANT] / "checkpoint.pt"

# Load only a checkpoint created by your own trusted training code.
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

assert isinstance(checkpoint, dict)
assert "model" in checkpoint
assert isinstance(checkpoint["model"], dict)

pretrained_weights = checkpoint["model"]
pretraining_steps = int(checkpoint.get("step", 0))

assert pretraining_steps > 0

print("Pretrained checkpoint")
print("---------------------")
print(f"Model: {VARIANT}")
print(f"Completed optimizer steps: {pretraining_steps:,}")
print(f"Saved parameter/buffer tensors: {len(pretrained_weights):,}")
print(f"Recorded objective: {checkpoint.get('objective', 'Not recorded')}")

print("\nCheckpoint loaded successfully.")
print("No model weights were modified.")


Pretrained checkpoint
---------------------
Model: ctddg_eq8
Completed optimizer steps: 171,920
Saved parameter/buffer tensors: 79
Recorded objective: Not recorded

Checkpoint loaded successfully.
No model weights were modified.


In [17]:
n_atom = config["n_atom"]
n_bond = config["n_bond"]
hidden = config["F_c"][-1]

expected_shapes = {
    "backbone.atom_embedding.weight": (n_atom, config["F_e"]),
    "backbone.mask_embedding.weight": (3, config["F_e"]),
    "init_atom_logits": (n_atom,),
    "rnn.weight_ih_l0": (3 * hidden, 2 * hidden),
    "rnn.weight_hh_l0": (3 * hidden, hidden),
    "policy.linear_x.weight": (
        n_bond * (n_atom + 1), config["F_policy"]
    ),
}

for i, width in enumerate(config["F_h"]):
    input_width = (
        config["F_e"] if i == 0 else config["F_h"][i - 1]
    )
    expected_shapes[f"backbone.convs.{i}.W"] = (
        input_width * (1 + n_bond + config["D"]),
        width
    )

for name, expected in expected_shapes.items():
    assert name in pretrained_weights, f"Missing: {name}"
    actual = tuple(pretrained_weights[name].shape)
    assert actual == expected, (
        f"{name}: Expected {expected}, found {actual}"
    )

# Check all saved floating-point tensors.
for name, tensor in pretrained_weights.items():
    assert isinstance(tensor, torch.Tensor), (
        f"{name}: Not a PyTorch tensor"
    )
    if torch.is_floating_point(tensor):
        assert torch.isfinite(tensor).all().item(), (
            f"{name}: Contains NaN or infinity"
        )

print("Pretrained weight verification")
print("------------------------------")
print(f"Architecture tensors checked: {len(expected_shapes)}")
print("Tensor shapes: Correct")
print("Missing required weights: None")
print("NaN or infinite values: None")

print("\nPretrained weights passed the initial checks.")


Pretrained weight verification
------------------------------
Architecture tensors checked: 12
Tensor shapes: Correct
Missing required weights: None
NaN or infinite values: None

Pretrained weights passed the initial checks.


## Reconstructing the Model and Transferring Weights

In [18]:
import torch.nn as nn
import torch.nn.functional as F

def sum_segments(x, segment_ids, count):
    result = x.new_zeros((count,) + x.shape[1:])
    return result.index_add(0, segment_ids, x)


class LinearBN(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.linear = nn.Linear(input_size, output_size, bias=False)
        self.bn = nn.BatchNorm1d(output_size)

    def forward(self, x):
        return self.bn(self.linear(x))


class GraphConv(nn.Module):
    def __init__(self, input_size, output_size, n_relations):
        super().__init__()

        self.W = nn.Parameter(
            torch.empty(input_size * (1 + n_relations), output_size)
        )
        nn.init.xavier_uniform_(self.W)
        self.n_relations = n_relations

    def forward(self, x, edge_lists):
        assert len(edge_lists) == self.n_relations

        terms = [x]

        for edges in edge_lists:
            if edges.numel() == 0:
                neighbor_features = torch.zeros_like(x)
            else:
                neighbor_features = sum_segments(
                    x[edges[1]], edges[0], x.shape[0]
                )

            terms.append(neighbor_features)

        return torch.cat(terms, dim=-1) @ self.W


# A small independent check.
test_conv = GraphConv(16, 32, 6)
assert test_conv.W.shape == (112, 32)

del test_conv

print("Graph computation layers are ready.")
print("Graph convolution supports six bond/distance relations.")

Graph computation layers are ready.
Graph convolution supports six bond/distance relations.


In [19]:
class GraphBackbone(nn.Module):
    def __init__(self, n_atom, n_bond, D, F_e, F_h, F_skip, F_c):
        super().__init__()

        self.atom_embedding = nn.Embedding(n_atom, F_e)
        self.mask_embedding = nn.Embedding(3, F_e)

        self.convs = nn.ModuleList()
        self.pre_bns = nn.ModuleList()

        for i, output_size in enumerate(F_h):
            input_size = F_e if i == 0 else F_h[i - 1]

            self.pre_bns.append(
                nn.Identity() if i == 0
                else nn.BatchNorm1d(input_size)
            )

            self.convs.append(
                GraphConv(input_size, output_size, n_bond + D)
            )

        total_width = sum(F_h)

        self.bn_skip = nn.BatchNorm1d(total_width)
        self.skip = LinearBN(total_width, F_skip)

        self.dense = nn.ModuleList()

        for input_size, output_size in zip(
            [F_skip] + list(F_c[:-1]), F_c
        ):
            self.dense.append(LinearBN(input_size, output_size))

    def forward(self, batch):
        x = (
            self.atom_embedding(batch["X"])
            + self.mask_embedding(batch["last_mask"])
        )

        layer_outputs = []

        for conv, bn in zip(self.convs, self.pre_bns):
            if not isinstance(bn, nn.Identity):
                x = F.relu(bn(x))

            x = conv(x, batch["edges"])
            layer_outputs.append(x)

        x = torch.cat(layer_outputs, dim=-1)
        x = F.relu(self.bn_skip(x))
        x = F.relu(self.skip(x))

        for dense in self.dense:
            x = dense(x)

        return x


print("Graph backbone defined successfully.")
print(f"GCN layers: {len(config['F_h'])}")
print(f"Final atom representation: {config['F_c'][-1]} dimensions")

Graph backbone defined successfully.
GCN layers: 6
Final atom representation: 512 dimensions


In [20]:
class Policy(nn.Module):
    def __init__(self, fin, hidden, n_atom, n_bond):
        super().__init__()

        self.n_atom = n_atom
        self.n_bond = n_bond

        self.linear_h = LinearBN(fin * 2, hidden)
        self.linear_h_t = LinearBN(fin, hidden)

        self.linear_x = nn.Linear(
            hidden, n_bond * (n_atom + 1)
        )
        self.linear_x_t = nn.Linear(hidden, 1)

    def forward(self, x, batch, x_end):
        node_to_graph = batch["NX_rep"]
        n_graphs = batch["NX"].shape[0]

        node_features = torch.cat(
            [x, x_end[node_to_graph]], dim=-1
        )

        node_logits = self.linear_x(
            F.relu(self.linear_h(node_features))
        )
        end_logits = self.linear_x_t(
            F.relu(self.linear_h_t(x_end))
        )

        # Joint normalization across every available graph action.
        node_max = node_logits.max(dim=-1).values
        graph_max = end_logits[:, 0].clone()
        graph_max = graph_max.scatter_reduce(
            0, node_to_graph, node_max,
            reduce="amax", include_self=True
        )

        node_weights = torch.exp(
            node_logits - graph_max[node_to_graph, None]
        )
        end_weights = torch.exp(
            end_logits - graph_max[:, None]
        )

        denominator = (
            sum_segments(
                node_weights.sum(-1, keepdim=True),
                node_to_graph, n_graphs
            )
            + end_weights
        )

        node_probs = node_weights / denominator[node_to_graph]
        end_probs = (end_weights / denominator)[:, 0]

        connect = node_probs[:, :self.n_bond]
        append = node_probs[:, self.n_bond:].reshape(
            -1, self.n_atom, self.n_bond
        )

        return append, connect, end_probs


class CTDDGPretrainer(nn.Module):
    def __init__(
        self, n_atom, n_bond, D=2, F_e=16,
        F_h=(32, 64, 128, 128, 256, 256),
        F_skip=256, F_c=(512,),
        F_policy=128, n_rnn=3
    ):
        super().__init__()

        self.n_atom = n_atom
        self.n_bond = n_bond

        self.backbone = GraphBackbone(
            n_atom, n_bond, D, F_e,
            list(F_h), F_skip, list(F_c)
        )

        self.rnn = nn.GRU(
            input_size=2 * F_c[-1],
            hidden_size=F_c[-1],
            num_layers=n_rnn,
            batch_first=True
        )

        self.policy = Policy(
            F_c[-1], F_policy, n_atom, n_bond
        )

        self.init_atom_logits = nn.Parameter(
            torch.zeros(n_atom)
        )

        # Match the existing pretrained model's initialization.
        for module in self.modules():
            if isinstance(module, (nn.Linear, nn.Embedding)):
                nn.init.xavier_uniform_(module.weight)

                if getattr(module, "bias", None) is not None:
                    nn.init.zeros_(module.bias)

            elif isinstance(module, nn.GRU):
                for name, parameter in module.named_parameters():
                    if "weight" in name:
                        nn.init.xavier_uniform_(parameter)
                    elif "bias" in name:
                        nn.init.zeros_(parameter)

            elif isinstance(module, nn.BatchNorm1d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)


# Construct the original unconditional model on CPU.
with torch.random.fork_rng(devices=[]):
    pretrained_model = CTDDGPretrainer(**config)

# Every saved parameter and buffer must match exactly.
pretrained_model.load_state_dict(
    pretrained_weights, strict=True
)

loaded_weights = pretrained_model.state_dict()

assert set(loaded_weights) == set(pretrained_weights)

for name, tensor in pretrained_weights.items():
    assert torch.equal(loaded_weights[name], tensor), (
        f"Pretrained weight mismatch: {name}"
    )

pretrained_model.eval()

parameter_count = sum(
    p.numel() for p in pretrained_model.parameters()
)

print("Pretrained model reconstruction")
print("-------------------------------")
print(f"Parameters: {parameter_count:,}")
print(f"Saved tensors: {len(pretrained_weights)}")
print("Checkpoint loading: Successful")
print("Transferred weights: All identical")

print("\nThe pretrained model architecture matches its checkpoint.")

Pretrained model reconstruction
-------------------------------
Parameters: 6,969,788
Saved tensors: 79
Checkpoint loading: Successful
Transferred weights: All identical

The pretrained model architecture matches its checkpoint.


In [21]:
class ConditionalGraphBackbone(GraphBackbone):
    def __init__(
        self, n_atom, n_bond, D, F_e,
        F_h, F_skip, F_c, n_c
    ):
        super().__init__(
            n_atom, n_bond, D, F_e, F_h, F_skip, F_c
        )

        self.n_c = n_c

        self.protein_projections = nn.ModuleList([
            nn.Linear(n_c, width, bias=False)
            for width in F_h
        ])

        for layer in self.protein_projections:
            nn.init.xavier_uniform_(layer.weight)

    def forward(self, batch):
        c = batch["c"]
        molecule_ids = batch["mol_ids_rep"]

        assert c.ndim == 2 and c.shape[1] == self.n_c
        assert molecule_ids.numel() == batch["X"].numel()

        x = (
            self.atom_embedding(batch["X"])
            + self.mask_embedding(batch["last_mask"])
        )

        outputs = []

        for conv, bn, projection in zip(
            self.convs,
            self.pre_bns,
            self.protein_projections
        ):
            x_input = (
                F.relu(bn(x))
                if not isinstance(bn, nn.Identity)
                else x
            )

            graph_features = conv(x_input, batch["edges"])
            protein_features = projection(c)[molecule_ids]

            x = graph_features + protein_features
            outputs.append(x)

        x = torch.cat(outputs, dim=-1)
        x = F.relu(self.bn_skip(x))
        x = F.relu(self.skip(x))

        for layer in self.dense:
            x = layer(x)

        return x


print("Protein-conditioned GCN defined.")
print(f"Protein projections: {len(config['F_h'])}")
print(f"Protein embedding size: {N_C:,}")


Protein-conditioned GCN defined.
Protein projections: 6
Protein embedding size: 6,165


In [22]:
class ConditionalCTDDG(CTDDGPretrainer):
    def __init__(self, model_config, n_c=6165):
        super().__init__(**model_config)

        self.backbone = ConditionalGraphBackbone(
            model_config["n_atom"],
            model_config["n_bond"],
            model_config["D"],
            model_config["F_e"],
            list(model_config["F_h"]),
            model_config["F_skip"],
            list(model_config["F_c"]),
            n_c
        )

        n_atom = model_config["n_atom"]
        head_width = 3 * n_atom

        self.init_head = nn.Sequential(
            nn.Linear(n_c, head_width, bias=False),
            nn.BatchNorm1d(head_width),
            nn.ReLU(),
            nn.Linear(head_width, n_atom)
        )

        for layer in self.init_head.modules():
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)

                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)


print("Conditional generator defined.")
print("GCN: Protein conditioning added")
print("GRU and action policy: Architecture preserved")
print("First-atom prediction: Protein-dependent network added")


Conditional generator defined.
GCN: Protein conditioning added
GRU and action policy: Architecture preserved
First-atom prediction: Protein-dependent network added


In [23]:
INIT_SEED = 42

# Initialize new layers reproducibly without changing
# the notebook's ongoing CPU random state.
with torch.random.fork_rng(devices=[]):
    torch.random.default_generator.manual_seed(INIT_SEED)
    model = ConditionalCTDDG(config, N_C)

missing, unexpected = model.load_state_dict(
    pretrained_weights, strict=False
)

allowed_prefixes = (
    "backbone.protein_projections.",
    "init_head."
)

assert not unexpected, f"Unexpected weights: {unexpected}"

assert all(
    name.startswith(allowed_prefixes)
    for name in missing
), f"Unexpected missing weights: {missing}"

new_keys = set(model.state_dict()) - set(pretrained_weights)

assert all(
    name.startswith(allowed_prefixes)
    for name in new_keys
), "Unexpected new model components."

assert set(pretrained_weights).issubset(model.state_dict())

for name, original in pretrained_weights.items():
    transferred = model.state_dict()[name]

    assert torch.equal(transferred, original), (
        f"Weight transfer failed: {name}"
    )

# Preserve the original first-atom distribution.
with torch.no_grad():
    model.init_head[-1].weight.zero_()
    model.init_head[-1].bias.copy_(
        F.log_softmax(model.init_atom_logits, dim=0)
    )

model.eval()

total_parameters = sum(
    p.numel() for p in model.parameters()
)

print("Pretrained weight transfer")
print("--------------------------")
print(f"Pretrained tensors preserved: {len(pretrained_weights)}")
print(f"New conditioning tensors: {len(new_keys)}")
print(f"Total model parameters: {total_parameters:,}")
print("Existing weights: Unchanged")
print("First-atom head: Initialized from pretraining")
print("\nWeight transfer verified successfully.")


Pretrained weight transfer
--------------------------
Pretrained tensors preserved: 79
New conditioning tensors: 14
Total model parameters: 13,097,481
Existing weights: Unchanged
First-atom head: Initialized from pretraining

Weight transfer verified successfully.


In [24]:
model.eval()

# Choose two different training proteins.
probe_ids = sorted(train_proteins)[:2]

assert len(probe_ids) == 2

protein_input = torch.from_numpy(
    protein_embeddings[probe_ids].copy()
)

with torch.no_grad():
    # Check the pretrained first-atom distribution.
    original_probs = F.softmax(
        pretrained_weights["init_atom_logits"], dim=0
    )

    conditional_probs = F.softmax(
        model.init_head(protein_input), dim=-1
    )

    assert torch.allclose(
        conditional_probs,
        original_probs.expand_as(conditional_probs),
        atol=1e-7,
        rtol=1e-6
    )

    # Check that every projection responds to protein input.
    differences = []

    for projection in model.backbone.protein_projections:
        output = projection(protein_input)

        assert torch.isfinite(output).all()

        difference = (
            output[0] - output[1]
        ).abs().max().item()

        differences.append(difference)

assert all(value > 0 for value in differences)

print("Protein conditioning checks")
print("---------------------------")
print("First-atom probabilities: Match pretraining")
print("Protein projections checked: 6 of 6")
print("Different proteins produce different projections: Yes")
print("Invalid numerical values: None")

print("\nInitial conditioning checks passed.")


Protein conditioning checks
---------------------------
First-atom probabilities: Match pretraining
Protein projections checked: 6 of 6
Different proteins produce different projections: Yes
Invalid numerical values: None

Initial conditioning checks passed.


## Graph Construction and Loss-Function Verification

In [25]:
def finetuning_loss(log_likelihood, toxicity, objective, tox_lambda=1.0):
    if log_likelihood.ndim != 1:
        raise ValueError("Expected one likelihood per molecule.")

    if toxicity.shape != log_likelihood.shape:
        raise ValueError("Toxicity labels do not match molecules.")

    if not torch.isfinite(log_likelihood).all():
        raise ValueError("Invalid molecular likelihood detected.")

    if not torch.all((toxicity == 0) | (toxicity == 1)):
        raise ValueError("Toxicity labels must be 0 or 1.")

    if objective == "cdgcn_nll":
        return -log_likelihood.mean()

    if objective == "ctddg_eq8":
        if tox_lambda < 0:
            raise ValueError("Toxicity weight cannot be negative.")

        weights = 1 - (1 + tox_lambda) * toxicity
        return -(log_likelihood * weights).mean()

    raise ValueError(f"Unknown objective: {objective}")


# Four example molecules: two non-toxic and two toxic.
example_ll = torch.tensor(
    [-2.0, -3.0, -4.0, -5.0],
    requires_grad=True
)

example_toxicity = torch.tensor([0, 1, 0, 1])

# Check CDGCN.
cdgcn_loss = finetuning_loss(
    example_ll, example_toxicity, "cdgcn_nll"
)

cdgcn_grad = torch.autograd.grad(
    cdgcn_loss, example_ll
)[0]

assert torch.allclose(cdgcn_grad, torch.full((4,), -0.25))
assert torch.allclose(cdgcn_loss, torch.tensor(3.5))

# Check CTDDG Equation 8.
ctddg_loss = finetuning_loss(
    example_ll, example_toxicity, "ctddg_eq8", tox_lambda=1.0
)

ctddg_grad = torch.autograd.grad(
    ctddg_loss, example_ll
)[0]

expected_grad = torch.tensor([-0.25, 0.25, -0.25, 0.25])

assert torch.allclose(ctddg_grad, expected_grad)
assert torch.allclose(ctddg_loss, torch.tensor(-0.5))

# Lambda = 0 excludes toxic molecules; it is not CDGCN.
zero_lambda_loss = finetuning_loss(
    example_ll, example_toxicity, "ctddg_eq8", tox_lambda=0.0
)

zero_lambda_grad = torch.autograd.grad(
    zero_lambda_loss, example_ll
)[0]

assert torch.allclose(
    zero_lambda_grad,
    torch.tensor([-0.25, 0.0, -0.25, 0.0])
)

print("Loss function verification")
print("--------------------------")
print("CDGCN: All molecules encouraged")
print("CTDDG: Non-toxic encouraged, toxic suppressed")
print("Toxicity weight: 1.0")
print("Gradient directions: Correct")
print("Lambda = 0: Toxic molecules excluded, not CDGCN")
print("\nBoth loss functions passed.")


Loss function verification
--------------------------
CDGCN: All molecules encouraged
CTDDG: Non-toxic encouraged, toxic suppressed
Toxicity weight: 1.0
Gradient directions: Correct
Lambda = 0: Toxic molecules excluded, not CDGCN

Both loss functions passed.


In [26]:

# --------------------------------------------------
# Select the fine-tuning experiment
# --------------------------------------------------

VARIANT = "cdgcn_nll"
FINETUNE_OBJECTIVE = "ctddg_eq8"
TOX_LAMBDA = 0.1

EXPERIMENT_NAME = "cdgcn_pretrain_to_ctddg_finetune"

RUN_TRAINING = False
EVALUATE_TEST = False
RUN_DIR = None

# --------------------------------------------------
# Load the CDGCN pretrained checkpoint
# --------------------------------------------------

checkpoint_path = PRETRAIN_RUNS[VARIANT] / "checkpoint.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

pretrained_weights = checkpoint["model"]
pretraining_steps = int(checkpoint["step"])
config = pretrain_metadata[VARIANT]["model_config"]

assert pretraining_steps > 0

# --------------------------------------------------
# Construct the conditional model
# --------------------------------------------------

with torch.random.fork_rng(devices=[]):
    torch.manual_seed(42)
    model = ConditionalCTDDG(config, N_C)

# --------------------------------------------------
# Transfer and verify pretrained weights
# --------------------------------------------------

missing, unexpected = model.load_state_dict(
    pretrained_weights,
    strict=False
)

model_state = model.state_dict()

new_keys = set(model_state) - set(pretrained_weights)
missing_keys = set(missing)

allowed_prefixes = (
    "backbone.protein_projections.",
    "init_head."
)

assert not unexpected, f"Unexpected weights: {unexpected}"

assert set(pretrained_weights).issubset(model_state)

assert all(
    name.startswith(allowed_prefixes)
    for name in new_keys
), f"Unexpected new layers: {new_keys}"

assert missing_keys.issubset(new_keys), (
    f"Unexpected missing weights: {missing_keys - new_keys}"
)

# BatchNorm counters may be initialized automatically.
automatically_initialized = new_keys - missing_keys

assert all(
    name.endswith(".num_batches_tracked")
    for name in automatically_initialized
), (
    "Unexpected unreported new entries: "
    f"{automatically_initialized}"
)

# All pretrained tensors must remain identical.
for name, original in pretrained_weights.items():
    assert torch.equal(model_state[name], original), (
        f"Pretrained weight mismatch: {name}"
    )

# --------------------------------------------------
# Initialize the protein-conditioned first-atom head
# --------------------------------------------------

with torch.no_grad():
    model.init_head[-1].weight.zero_()
    model.init_head[-1].bias.copy_(
        F.log_softmax(model.init_atom_logits, dim=0)
    )

model.eval()

probe_ids = sorted(train_proteins)[:2]

protein_input = torch.from_numpy(
    protein_embeddings[probe_ids].copy()
)

with torch.no_grad():
    original_probs = F.softmax(
        model.init_atom_logits, dim=0
    )

    conditional_probs = F.softmax(
        model.init_head(protein_input), dim=-1
    )

    assert torch.allclose(
        conditional_probs,
        original_probs.expand_as(conditional_probs),
        atol=1e-7,
        rtol=1e-6
    )

# --------------------------------------------------
# Summary
# --------------------------------------------------

print("Fine-tuning experiment")
print("----------------------")
print("Pretraining: CDGCN (without toxicity)")
print("Fine-tuning: CTDDG toxicity-aware loss")
print(f"Toxicity weight: {TOX_LAMBDA}")
print(f"Pretraining steps: {pretraining_steps:,}")

print("\nModel initialization")
print("--------------------")
print(f"Pretrained tensors preserved: {len(pretrained_weights)}")
print(f"New conditioning entries: {len(new_keys)}")
print("Weight transfer: Verified")
print("First-atom initialization: Verified")

print("\nModel is ready for further checks.")
print("Training has not started.")


Fine-tuning experiment
----------------------
Pretraining: CDGCN (without toxicity)
Fine-tuning: CTDDG toxicity-aware loss
Toxicity weight: 0.1
Pretraining steps: 687,695

Model initialization
--------------------
Pretrained tensors preserved: 79
New conditioning entries: 14
Weight transfer: Verified
First-atom initialization: Verified

Model is ready for further checks.
Training has not started.


In [27]:
import math

K_PATHS = int(pretrain_metadata[VARIANT]["k_paths"])
ALPHA = float(pretrain_metadata[VARIANT]["alpha"])

def molecular_log_likelihood(path_log_p, path_log_q):
    if path_log_p.ndim != 2 or path_log_p.shape != path_log_q.shape:
        raise ValueError("Path probabilities must have matching [batch, K] shapes.")

    if not torch.isfinite(path_log_p).all():
        raise ValueError("Invalid model log probabilities.")

    if not torch.isfinite(path_log_q).all() or (path_log_q > 0).any():
        raise ValueError("Invalid path-sampling log probabilities.")

    k = path_log_p.shape[1]
    assert k > 0

    corrected = path_log_p - path_log_q

    return torch.logsumexp(corrected, dim=1) - math.log(k)


# Two example molecules, with five sampled paths each.
assert K_PATHS == 5
assert ALPHA == 0.8

path_log_p = torch.tensor([
    [-15., -16., -18., -19., -20.],
    [ -6.,  -7.,  -8., -10., -11.]
], dtype=torch.float64, requires_grad=True)

# Illustrative path probabilities for a two-choice branch.
path_q = torch.tensor([
    [0.8, 0.2, 0.8, 0.2, 0.8],
    [0.2, 0.8, 0.8, 0.2, 0.8]
], dtype=torch.float64)

path_log_q = path_q.log()

# Calculate one likelihood per molecule.
log_likelihood = molecular_log_likelihood(
    path_log_p, path_log_q
)

assert log_likelihood.shape == (2,)

# Compare against the direct mathematical calculation.
direct_result = (
    (path_log_p.exp() / path_q).mean(dim=1)
).log()

assert torch.allclose(log_likelihood, direct_result, atol=1e-10)

# Verify stability for extremely small probabilities.
extreme_log_p = torch.full((1, K_PATHS), -1000.0)

extreme_result = molecular_log_likelihood(
    extreme_log_p, path_log_q[:1].float()
)

assert torch.isfinite(extreme_result).all()

# Check gradients when this likelihood enters CTDDG Eq. 8.
labels = torch.tensor([0, 1])

loss = finetuning_loss(
    log_likelihood,
    labels,
    objective=FINETUNE_OBJECTIVE,
    tox_lambda=TOX_LAMBDA
)

gradients = torch.autograd.grad(loss, path_log_p)[0]

assert torch.isfinite(gradients).all()
assert (gradients[0] < 0).all()
assert (gradients[1] > 0).all()

print("Molecular likelihood verification")
print("---------------------------------")
print(f"Generation paths per molecule: {K_PATHS}")
print(f"Path sampling alpha: {ALPHA}")
print("Likelihood calculation: Correct")
print("Very small probabilities: Handled safely")
print("Toxicity-aware gradients: Correct")

print("\nExample molecular log-likelihoods:")
print(f"  Non-toxic molecule: {log_likelihood[0].item():.2f}")
print(f"  Toxic molecule: {log_likelihood[1].item():.2f}")

print("\nAll likelihood checks passed.")


Molecular likelihood verification
---------------------------------
Generation paths per molecule: 5
Path sampling alpha: 0.8
Likelihood calculation: Correct
Very small probabilities: Handled safely
Toxicity-aware gradients: Correct

Example molecular log-likelihoods:
  Non-toxic molecule: -15.43
  Toxic molecule: -5.86

All likelihood checks passed.


In [28]:
# Use the existing model without changing its weights.
model.eval()
device = next(model.parameters()).device

# Two small example graphs with 2 and 3 atoms.
node_counts = torch.tensor([2, 3], device=device)
graph_ids = torch.repeat_interleave(
    torch.arange(2, device=device), node_counts
)

# Bidirectional edges within each example graph.
bond_edges = torch.tensor([
    [0, 1, 2, 3, 3, 4],
    [1, 0, 3, 2, 4, 3]
], dtype=torch.long, device=device)

n_relations = config["n_bond"] + config["D"]

edges = [bond_edges] + [
    torch.empty((2, 0), dtype=torch.long, device=device)
    for _ in range(n_relations - 1)
]

protein_ids = sorted(train_proteins)[:2]

toy_batch = {
    "X": torch.tensor([0, 1, 2, 3, 4], device=device),
    "last_mask": torch.tensor([0, 1, 0, 0, 1], device=device),
    "edges": edges,
    "NX": node_counts,
    "NX_rep": graph_ids,
    "mol_ids_rep": graph_ids,
    "c": torch.as_tensor(
        protein_embeddings[protein_ids].copy(),
        dtype=torch.float32,
        device=device
    )
}

with torch.no_grad():
    # Generate atom representations using the actual GCN.
    atom_features = model.backbone(toy_batch)

    # Build graph-level and last-atom representations.
    graph_features = (
        sum_segments(atom_features, graph_ids, 2)
        / node_counts[:, None]
    )

    last_atoms = torch.cumsum(node_counts, dim=0) - 1
    last_features = atom_features[last_atoms]

    # Process one generation step through the actual GRU.
    rnn_input = torch.cat(
        [graph_features, last_features], dim=-1
    ).unsqueeze(1)

    rnn_output, _ = model.rnn(rnn_input)
    graph_states = rnn_output[:, 0, :]

    # Calculate the actual action probabilities.
    append, connect, end = model.policy(
        atom_features, toy_batch, graph_states
    )

    # Combine all possible actions for each graph.
    node_probabilities = (
        append.sum(dim=(1, 2)) + connect.sum(dim=1)
    )

    total_probabilities = (
        sum_segments(
            node_probabilities[:, None], graph_ids, 2
        )[:, 0] + end
    )

    # The first-atom distribution must also be normalized.
    init_probs = F.softmax(
        model.init_head(toy_batch["c"]), dim=-1
    )

# Verify output dimensions.
assert append.shape == (
    5, config["n_atom"], config["n_bond"]
)
assert connect.shape == (5, config["n_bond"])
assert end.shape == (2,)
assert init_probs.shape == (2, config["n_atom"])

# All probabilities must be finite and between 0 and 1.
for probabilities in (append, connect, end, init_probs):
    assert torch.isfinite(probabilities).all()
    assert ((probabilities >= 0) & (probabilities <= 1)).all()

# All available actions must sum to 1 for each graph.
assert torch.allclose(
    total_probabilities,
    torch.ones_like(total_probabilities),
    atol=1e-5
)

assert torch.allclose(
    init_probs.sum(dim=1),
    torch.ones(2, device=device),
    atol=1e-5
)

print("Graph action probability verification")
print("-------------------------------------")
print("Example graphs tested: 2")
print("GCN and GRU forward pass: Successful")
print("Append, connect and end probabilities: Valid")
print("Action probabilities sum to 1: Yes")
print("First-atom probabilities sum to 1: Yes")
print("Invalid numerical values: None")

print("\nBoth graphs passed probability normalization checks.")


Graph action probability verification
-------------------------------------
Example graphs tested: 2
GCN and GRU forward pass: Successful
Append, connect and end probabilities: Valid
Action probabilities sum to 1: Yes
First-atom probabilities sum to 1: Yes
Invalid numerical values: None

Both graphs passed probability normalization checks.


## Real BindingDB Training Pipeline

### Molecular graph representation

In [29]:

import random
import math
import numpy as np
import networkx as nx

from scipy import sparse
from rdkit import Chem

MAX_ITER_PER_MOL = 120
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32

In [30]:
class MoleculeSpec:
    def __init__(self, path):
        self.atom_types = []

        for line in Path(path).read_text().splitlines():
            if line.strip():
                symbol, charge, hydrogens = line.strip().split(",")
                self.atom_types.append(
                    (symbol, int(charge), int(hydrogens))
                )

        self.atom_index = {
            atom: i for i, atom in enumerate(self.atom_types)
        }

        self.bond_orders = [
            Chem.BondType.AROMATIC,
            Chem.BondType.SINGLE,
            Chem.BondType.DOUBLE,
            Chem.BondType.TRIPLE
        ]

        self.bond_index = {
            bond: i for i, bond in enumerate(self.bond_orders)
        }

        self.num_atom_types = len(self.atom_types)
        self.num_bond_types = len(self.bond_orders)
        self.max_iter = MAX_ITER_PER_MOL

    def get_atom_type(self, atom):
        key = (
            atom.GetSymbol(),
            atom.GetFormalCharge(),
            atom.GetNumExplicitHs()
        )
        return self.atom_index[key]

    def get_bond_type(self, bond):
        return self.bond_index[bond.GetBondType()]

In [31]:
mol_spec = MoleculeSpec(ATOM_TYPES_FILE)

assert mol_spec.num_atom_types == config["n_atom"]
assert mol_spec.num_bond_types == config["n_bond"]


def get_graph_from_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError(f"Invalid molecule: {smiles}")

    atom_types = [
        mol_spec.get_atom_type(a) for a in mol.GetAtoms()
    ]

    ranks = list(Chem.CanonicalRankAtoms(mol))

    bonds = [
        [b.GetBeginAtomIdx(), b.GetEndAtomIdx()]
        for b in mol.GetBonds()
    ]

    bond_types = [
        mol_spec.get_bond_type(b) for b in mol.GetBonds()
    ]

    graph = nx.Graph()
    graph.add_nodes_from(range(len(atom_types)))
    graph.add_edges_from(bonds)

    if not nx.is_connected(graph):
        raise ValueError("Disconnected molecular graph.")

    return graph, atom_types, ranks, bonds, bond_types

In [32]:
def traverse_graph(graph, ranks, current_node=None,
                   step_ids=None, p=0.8, log_p=0.0):

    if current_node is None:
        next_nodes = list(range(len(ranks)))
        step_ids = [-1] * len(ranks)
    else:
        next_nodes = [
            n for n in graph.neighbors(current_node)
            if step_ids[n] < 0
        ]

    next_nodes.sort(key=lambda n: ranks[n])

    while next_nodes:
        if len(next_nodes) == 1:
            next_node = next_nodes[0]

        elif random.random() >= 1 - p:
            next_node = next_nodes[0]
            log_p += math.log(p)

        else:
            next_node = next_nodes[
                random.randint(1, len(next_nodes) - 1)
            ]
            log_p += math.log(
                (1 - p) / (len(next_nodes) - 1)
            )

        step_ids[next_node] = max(step_ids) + 1

        _, log_p = traverse_graph(
            graph, ranks, next_node, step_ids, p, log_p
        )

        next_nodes = [
            n for n in next_nodes if step_ids[n] < 0
        ]

    return step_ids, log_p

In [33]:
def single_reorder(X, A, step_ids):
    X, A = X.copy(), A.copy()

    if len(A) == 0:
        assert len(X) == 1
        return X, np.empty((0, 3), dtype=np.int32)

    step_ids = np.asarray(step_ids, dtype=np.int32)

    X = X[np.argsort(step_ids)]
    A[:, :2] = step_ids[A[:, :2]]

    maximum = A[:, :2].max(axis=1)
    minimum = A[:, :2].min(axis=1)

    A = A[np.lexsort([-minimum, maximum])]

    maximum = A[:, :2].max(axis=1)
    minimum = A[:, :2].min(axis=1)

    is_append = np.concatenate([
        [True], maximum[1:] > maximum[:-1]
    ])

    A[:, :2] = np.where(
        is_append[:, None],
        np.stack([minimum, maximum], axis=1),
        np.stack([maximum, minimum], axis=1)
    )

    return X, A


print("Molecular graph processing is ready.")
print(f"Atom types: {mol_spec.num_atom_types}")
print(f"Bond types: {mol_spec.num_bond_types}")
print(f"Maximum generation steps: {MAX_ITER_PER_MOL}")

Molecular graph processing is ready.
Atom types: 43
Bond types: 4
Maximum generation steps: 120


In [34]:

def single_expand(X0, A0):
    X0, A0 = X0.copy(), A0.copy()

    is_append = (A0[:, 0] < A0[:, 1]).astype(np.int32)

    NX = np.cumsum(
        np.pad(is_append, (1, 0), constant_values=1)
    )

    shift = np.cumsum(
        np.pad(NX, (1, 0))[:-1]
    )

    X_index = (
        np.arange(NX.sum())
        - np.repeat(shift, NX)
    )

    X = X0[X_index]

    _, A_index = np.tril_indices(len(A0))
    A = A0[A_index]

    NA = np.arange(len(A0) + 1)

    action_type = 1 - is_append

    actions = np.stack([
        action_type,
        np.where(action_type == 0, X0[A0[:, 1]], 0),
        A0[:, 2],
        np.where(action_type == 0, A0[:, 0], 0),
        np.where(action_type == 1, A0[:, 1], 0)
    ], axis=1)

    actions = np.concatenate([
        actions,
        np.array([[2, 0, 0, 0, 0]], dtype=np.int32)
    ])

    mask = np.zeros(len(X), dtype=np.int32)

    mask[shift + NX - 1] = np.where(
        np.pad(is_append, (1, 0), constant_values=1) == 1,
        1, 2
    )

    return (
        np.array([X0[0]], dtype=np.int32),
        X, NX, A, NA, actions, mask
    )


def get_d(A, X):
    def adjacency(edges):
        src = np.concatenate([edges[:, 0], edges[:, 1]])
        dst = np.concatenate([edges[:, 1], edges[:, 0]])

        return sparse.coo_matrix(
            (np.ones(len(src), dtype=np.int32), (src, dst)),
            shape=(len(X), len(X))
        )

    base = adjacency(A)
    d2 = base @ base
    d3 = d2 @ base

    D2 = np.stack(d2.nonzero(), axis=1)
    D2 = D2[D2[:, 0] < D2[:, 1]]

    D3 = np.stack(d3.nonzero(), axis=1)
    D3 = D3[D3[:, 0] < D3[:, 1]]

    d3_sparse = adjacency(D3)
    d3_sparse = d3_sparse - d3_sparse.multiply(base)

    D3 = np.stack(d3_sparse.nonzero(), axis=1)
    D3 = D3[D3[:, 0] < D3[:, 1]]

    return D2, D3


def merge_single(X, A, NX, NA, mol_ids,
                 rep_ids, iw_ids, action_0,
                 actions, mask, log_p):

    offsets = np.cumsum(np.pad(NX, (1, 0))[:-1])
    A[:, :2] += np.repeat(offsets, NA)[:, None]

    D2, D3 = get_d(A, X)

    edges = [
        A[A[:, 2] == i, :2]
        for i in range(mol_spec.num_bond_types)
    ]

    edges.extend([D2, D3])

    NX_rep = np.repeat(np.arange(len(NX)), NX)

    actions[:, 3] += offsets * (actions[:, 0] == 0)
    actions[:, 4] += offsets * (actions[:, 0] == 1)

    mol_ids_rep = np.repeat(mol_ids, NX)
    rep_ids_rep = np.repeat(rep_ids, NX)

    return (
        X, edges, mol_ids_rep, rep_ids_rep,
        iw_ids, mask, NX, NX_rep,
        action_0, actions, log_p
    )


def process_single(record, k=5, p=0.8):
    smiles, label = record.rsplit(maxsplit=1)
    assert label in ("0", "1")

    graph, atoms, ranks, bonds, bond_types = (
        get_graph_from_smiles(smiles)
    )

    X0 = np.asarray(atoms, dtype=np.int32)

    A0 = np.column_stack([
        np.asarray(bonds, dtype=np.int32).reshape(-1, 2),
        np.asarray(bond_types, dtype=np.int32)
    ])

    if len(bonds) + 1 > MAX_ITER_PER_MOL:
        raise ValueError("Molecule exceeds generation limit.")

    fields = [[] for _ in range(11)]

    for path in range(k):
        steps, log_q = traverse_graph(graph, ranks, p=p)
        X, A = single_reorder(X0, A0, steps)

        init, X, NX, A, NA, actions, mask = (
            single_expand(X, A)
        )

        values = [
            X, A, NX, NA,
            np.zeros(len(NX), dtype=np.int32),
            np.full(len(NX), path, dtype=np.int32),
            np.full(len(NX), path, dtype=np.int32),
            init, actions, mask,
            np.array([log_q], dtype=np.float32)
        ]

        for destination, value in zip(fields, values):
            destination.append(value)

    return tuple(
        np.concatenate(values) for values in fields
    ) + (np.array([int(label)], dtype=np.int32),)


print("Generation-path processing is ready.")
print("Append, connect and end actions are supported.")
print("Importance-sampling path probabilities are recorded.")


Generation-path processing is ready.
Append, connect and end actions are supported.
Importance-sampling path probabilities are recorded.


In [35]:

def collate_graph_records(records, k=5, alpha=0.8):
    fields = [[] for _ in range(12)]
    steps_per_molecule = []

    for i, record in enumerate(records):
        parts = list(process_single(record, k, alpha))

        parts[4] += i
        parts[6] += i * k

        for j, value in enumerate(parts):
            fields[j].append(value)

        smiles = record.rsplit(maxsplit=1)[0]
        steps_per_molecule.append(
            Chem.MolFromSmiles(smiles).GetNumBonds() + 1
        )

    (
        X, A, NX, NA, mol_ids, rep_ids, iw_ids,
        action_0, actions, mask, log_q, toxicity
    ) = [
        np.concatenate(values)
        for values in fields
    ]

    (
        X, A, mol_ids_rep, rep_ids_rep, iw_ids,
        mask, NX, NX_rep, action_0, actions, log_q
    ) = merge_single(
        X, A, NX, NA, mol_ids, rep_ids,
        iw_ids, action_0, actions, mask, log_q
    )

    B = len(records)

    graph_to_rnn = np.zeros(
        (B, k, MAX_ITER_PER_MOL), dtype=np.int64
    )

    rnn_to_graph = []
    offset = 0

    for i, steps in enumerate(steps_per_molecule):
        for path in range(k):
            graph_to_rnn[i, path, :steps] = (
                np.arange(steps) + offset
            )

            rnn_to_graph.extend(
                (i, path, t) for t in range(steps)
            )

            offset += steps

    assert offset == len(NX)

    edge_tensors = []

    for edge_array in A:
        if len(edge_array) == 0:
            tensor = torch.empty(
                (2, 0), dtype=torch.long, device=DEVICE
            )
        else:
            both = np.concatenate([
                edge_array[:, :2],
                edge_array[:, [1, 0]]
            ])

            tensor = torch.as_tensor(
                both.T.copy(), dtype=torch.long, device=DEVICE
            )

        edge_tensors.append(tensor)

    def tensor(values, dtype=torch.long):
        return torch.as_tensor(
            values, dtype=dtype, device=DEVICE
        )

    batch = {
        "X": tensor(X),
        "edges": edge_tensors,
        "iw_ids": tensor(iw_ids),
        "last_mask": tensor(mask),
        "NX": tensor(NX),
        "NX_rep": tensor(NX_rep),
        "mol_ids_rep": tensor(mol_ids_rep),
        "action_0": tensor(action_0),
        "actions": tensor(actions),
        "log_p_sigma": tensor(log_q, torch.float32),
        "tox_class": tensor(toxicity),
        "graph_to_rnn": tensor(graph_to_rnn),
        "rnn_to_graph": tensor(
            np.asarray(rnn_to_graph, dtype=np.int64)
        ),
        "NX_cum": tensor(np.cumsum(NX)),
        "steps_per_molecule": tensor(
            steps_per_molecule, torch.float32
        ),
        "batch_size": B,
        "iw_size": k
    }

    assert batch["action_0"].numel() == B * k
    assert batch["log_p_sigma"].numel() == B * k
    assert batch["tox_class"].numel() == B

    return batch


def attach_proteins(rows):
    records = [
        f"{r['smiles']} {int(r['ames'])}"
        for r in rows
    ]

    batch = collate_graph_records(
        records, k=K_PATHS, alpha=ALPHA
    )

    indices = np.array([
        int(r["protein_index"]) for r in rows
    ])

    assert (indices >= 0).all()
    assert (indices < len(protein_embeddings)).all()

    batch["c"] = torch.as_tensor(
        np.ascontiguousarray(protein_embeddings[indices]),
        dtype=torch.float32,
        device=DEVICE
    )

    return batch


# One real non-toxic and one real toxic training example.
example_rows = [
    splits["train"].loc[splits["train"]["ames"] == label]
    .iloc[0].to_dict()
    for label in (0, 1)
]

random.seed(42)
batch = attach_proteins(example_rows)

assert batch["c"].shape == (2, N_C)
assert batch["tox_class"].tolist() == [0, 1]
assert torch.isin(
    batch["actions"][:, 0],
    torch.tensor([0, 1, 2], device=DEVICE)
).all()

print("Real BindingDB batch")
print("--------------------")
print("Molecules: 2")
print("Non-toxic: 1")
print("Predicted toxic: 1")
print(f"Generation paths: {batch['action_0'].numel()}")
print(f"Protein embedding size: {batch['c'].shape[1]:,}")
print("Graph actions: Valid")
print("\nBatch preparation passed.")


Real BindingDB batch
--------------------
Molecules: 2
Non-toxic: 1
Predicted toxic: 1
Generation paths: 10
Protein embedding size: 6,165
Graph actions: Valid

Batch preparation passed.


In [36]:

def real_molecular_loglik(model, batch):
    B = batch["batch_size"]
    K = batch["iw_size"]

    X = model.backbone(batch)
    n_steps = batch["NX"].numel()

    graph_mean = (
        sum_segments(X, batch["NX_rep"], n_steps)
        / batch["NX"][:, None].to(X.dtype)
    )

    last_atom = X[batch["NX_cum"] - 1]
    features = torch.cat([graph_mean, last_atom], dim=-1)

    padded = features[batch["graph_to_rnn"]]
    padded = padded.reshape(B * K, -1, features.shape[-1])

    rnn_output, _ = model.rnn(padded)
    rnn_output = rnn_output.reshape(B, K, -1, rnn_output.shape[-1])

    indices = batch["rnn_to_graph"]

    graph_states = rnn_output[
        indices[:, 0], indices[:, 1], indices[:, 2]
    ]

    append, connect, end = model.policy(
        X, batch, graph_states
    )

    actions = batch["actions"]
    action_type = actions[:, 0]

    initial_ll = F.log_softmax(
        model.init_head(batch["c"]), dim=-1
    )

    initial_ll = initial_ll.repeat_interleave(K, dim=0)[
        torch.arange(B * K, device=DEVICE),
        batch["action_0"]
    ]

    # Preserve the original model's probability-floor behavior.
    eps = 1e-10

    append_ll = torch.log(
        append[actions[:, 3], actions[:, 1], actions[:, 2]] + eps
    )

    connect_ll = torch.log(
        connect[actions[:, 4], actions[:, 2]] + eps
    )

    end_ll = torch.log(end + eps)

    action_ll = torch.where(
        action_type == 0,
        append_ll,
        torch.where(action_type == 1, connect_ll, end_ll)
    )

    path_ll = (
        sum_segments(
            action_ll[:, None], batch["iw_ids"], B * K
        )[:, 0] + initial_ll
    )

    corrected_ll = (
        path_ll.reshape(B, K)
        - batch["log_p_sigma"].reshape(B, K)
    )

    return torch.logsumexp(corrected_ll, dim=1) - math.log(K)


# Run on the allocated GPU.
model = model.to(DEVICE)

# GRU backward needs train mode; keep BatchNorm statistics fixed
# during this diagnostic so the test doesn't alter their buffers.
model.train()

for layer in model.modules():
    if isinstance(layer, nn.modules.batchnorm._BatchNorm):
        layer.eval()

model.zero_grad(set_to_none=True)

ll = real_molecular_loglik(model, batch)

assert ll.shape == (2,)
assert torch.isfinite(ll).all()

loss = finetuning_loss(
    ll,
    batch["tox_class"],
    FINETUNE_OBJECTIVE,
    TOX_LAMBDA
)

assert torch.isfinite(loss)

loss.backward()

# Verify that training signals reach the protein projections.
projection_grads = [
    p.grad for p in model.backbone.protein_projections.parameters()
]

assert all(
    g is not None and torch.isfinite(g).all()
    for g in projection_grads
)

assert any(g.abs().sum().item() > 0 for g in projection_grads)

# Count total gradient magnitude for diagnostics.
grad_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(), max_norm=float("inf")
).item()

assert math.isfinite(grad_norm)

print("Real GPU training test")
print("----------------------")
print(f"Non-toxic log-likelihood: {ll[0].item():.2f}")
print(f"Toxic log-likelihood: {ll[1].item():.2f}")
print(f"CTDDG loss: {loss.item():.2f}")
print(f"Gradient norm: {grad_norm:.2f}")
print("Protein-conditioning gradients: Valid")
print("Forward pass: Passed")
print("Backward pass: Passed")

model.zero_grad(set_to_none=True)
model.eval()

print("\nReady for optimizer and training checks.")


Real GPU training test
----------------------
Non-toxic log-likelihood: -47.20
Toxic log-likelihood: -45.69
CTDDG loss: 21.31
Gradient norm: 203.93
Protein-conditioning gradients: Valid
Forward pass: Passed
Backward pass: Passed

Ready for optimizer and training checks.


## Optimizer Smoke Test

In [37]:
import copy

LEARNING_RATE = 1e-4
GRAD_CLIP = 3.0
BATCH_SIZE = 32

# Select 32 real training examples.
sample_rows = (
    splits["train"]
    .sample(n=BATCH_SIZE, random_state=42)
    .to_dict("records")
)

assert {int(r["ames"]) for r in sample_rows} == {0, 1}

# Build a real training batch.
smoke_batch = attach_proteins(sample_rows)

# Work on a separate model so original weights stay unchanged.
smoke_model = copy.deepcopy(model).to(DEVICE)
smoke_model.train()

optimizer = torch.optim.Adam(
    smoke_model.parameters(),
    lr=LEARNING_RATE
)

# Remember weights before updating.
before = {
    "gcn": smoke_model.backbone.convs[0].W.detach().clone(),
    "protein": smoke_model.backbone.protein_projections[0]
        .weight.detach().clone()
}

# One real training step.
optimizer.zero_grad(set_to_none=True)

ll = real_molecular_loglik(smoke_model, smoke_batch)

loss = finetuning_loss(
    ll,
    smoke_batch["tox_class"],
    FINETUNE_OBJECTIVE,
    TOX_LAMBDA
)

assert torch.isfinite(ll).all()
assert torch.isfinite(loss)

loss.backward()

grad_norm = torch.nn.utils.clip_grad_norm_(
    smoke_model.parameters(),
    float("inf"),
    error_if_nonfinite=True
)

assert grad_norm.item() > 0
assert torch.isfinite(grad_norm)

# Apply the original element-wise gradient clipping.
torch.nn.utils.clip_grad_value_(
    smoke_model.parameters(),
    GRAD_CLIP
)

optimizer.step()

# Check whether important model components were updated.
gcn_change = (
    smoke_model.backbone.convs[0].W - before["gcn"]
).abs().max().item()

protein_change = (
    smoke_model.backbone.protein_projections[0].weight
    - before["protein"]
).abs().max().item()

assert gcn_change > 0
assert protein_change > 0

# Verify the updated model still produces finite likelihoods.
smoke_model.eval()

with torch.no_grad():
    updated_ll = real_molecular_loglik(
        smoke_model, smoke_batch
    )

assert torch.isfinite(updated_ll).all()

print("Optimizer smoke test")
print("--------------------")
print(f"Molecules: {BATCH_SIZE}")
print(f"Predicted toxic: {smoke_batch['tox_class'].sum().item()}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Initial CTDDG loss: {loss.item():.2f}")
print(f"Gradient norm before clipping: {grad_norm.item():.2f}")
print(f"Gradient clipping: {GRAD_CLIP} per element")
print("GCN weights updated: Yes")
print("Protein-conditioning weights updated: Yes")
print("Updated model outputs: Finite")

# Discard the temporary training model.
del smoke_model, optimizer, smoke_batch, before
torch.cuda.empty_cache()

model.eval()

print("\nOptimizer test passed. Original model is unchanged.")


Optimizer smoke test
--------------------
Molecules: 32
Predicted toxic: 11
Learning rate: 0.0001
Initial CTDDG loss: 23.21
Gradient norm before clipping: 23.05
Gradient clipping: 3.0 per element
GCN weights updated: Yes
Protein-conditioning weights updated: Yes
Updated model outputs: Finite

Optimizer test passed. Original model is unchanged.


## Full BindingDB Fine-tuning

In [38]:
# Experiment
VARIANT = "cdgcn_nll"
FINETUNE_OBJECTIVE = "ctddg_eq8"
TOX_LAMBDA = 0.1
SEED = 42

# Training
BATCH_SIZE = 32
K_PATHS = 5
ALPHA = 0.8
MAX_EPOCHS = 10
LEARNING_RATE = 1e-4
LR_DECAY = 0.002
LR_DECAY_STEP = 100
GRAD_CLIP = 3.0

# Validation and checkpointing
PATIENCE = 3
MIN_DELTA = 0.01
LOG_EVERY = 50
SAVE_EVERY = 200
MONITOR_EVERY = 200
MONITOR_VAL_PAIRS = 256

# Safety
EXPERIMENT_CLASS = "exploratory_leaked_pretraining"
EVALUATE_TEST = False
RUN_TRAINING = False

EXPERIMENT_NAME = "cdgcn_to_ctddg_d1_bs32_lambda0p1_v1"

RUN_DIR = (
    REPO_ROOT
    / "REWORKED_CODE/finetuning/finetuning_runs"
    / EXPERIMENT_NAME
)

# Verify the selected experiment.
assert VARIANT == "cdgcn_nll"
assert FINETUNE_OBJECTIVE == "ctddg_eq8"
assert TOX_LAMBDA == 0.1
assert K_PATHS == 5
assert ALPHA == 0.8
assert len(splits["train"]) == 118765
assert len(splits["val"]) == 6675
assert not EVALUATE_TEST
assert not RUN_TRAINING

# Never overwrite an existing experiment accidentally.
existing_run = RUN_DIR.exists() and any(RUN_DIR.iterdir())

print("Fine-tuning configuration")
print("-------------------------")
print("Starting weights: CDGCN (without toxicity)")
print("Training objective: CTDDG Equation 8")
print(f"Toxicity weight: {TOX_LAMBDA}")
print(f"Training pairs: {len(splits['train']):,}")
print(f"Validation pairs: {len(splits['val']):,}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Maximum epochs: {MAX_EPOCHS}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Gradient clipping: {GRAD_CLIP} per element")

print("\nCheckpoint location:")
print(RUN_DIR)

if existing_run:
    print("\nExisting run detected. Resume verification is required.")
else:
    print("\nNew experiment directory is available.")

print("\nConfiguration verified. Training has not started.")


Fine-tuning configuration
-------------------------
Starting weights: CDGCN (without toxicity)
Training objective: CTDDG Equation 8
Toxicity weight: 0.1
Training pairs: 118,765
Validation pairs: 6,675
Batch size: 32
Maximum epochs: 10
Learning rate: 0.0001
Gradient clipping: 3.0 per element

Checkpoint location:
/home/muhamed/repo/CTDDG/REWORKED_CODE/finetuning/finetuning_runs/cdgcn_to_ctddg_d1_bs32_lambda0p1_v1

Existing run detected. Resume verification is required.

Configuration verified. Training has not started.


In [39]:

import random
import numpy as np

train_rows = splits["train"].to_dict("records")
val_rows = splits["val"].to_dict("records")

train_lengths = [len(row["smiles"]) for row in train_rows]

def make_training_batches(epoch):
    rng = random.Random(SEED + epoch)

    indices = list(range(len(train_rows)))
    rng.shuffle(indices)

    # Group similarly sized molecules to reduce memory usage.
    pool_size = BATCH_SIZE * 32
    batches = []

    for start in range(0, len(indices), pool_size):
        pool = indices[start:start + pool_size]
        pool.sort(key=lambda i: train_lengths[i])

        for j in range(0, len(pool), BATCH_SIZE):
            batches.append(pool[j:j + BATCH_SIZE])

    rng.shuffle(batches)
    return batches


# Verify that every training pair appears exactly once.
example_plan = make_training_batches(0)
all_indices = [i for batch_ids in example_plan for i in batch_ids]

assert sorted(all_indices) == list(range(len(train_rows)))
assert all(1 <= len(b) <= BATCH_SIZE for b in example_plan)

print("Training data preparation")
print("-------------------------")
print(f"Training pairs: {len(train_rows):,}")
print(f"Validation pairs: {len(val_rows):,}")
print(f"Batches per epoch: {len(example_plan):,}")
print("Every training pair included: Yes")
print("Batch ordering: Reproducible")

print("\nTraining data is ready.")


Training data preparation
-------------------------
Training pairs: 118,765
Validation pairs: 6,675
Batches per epoch: 3,712
Every training pair included: Yes
Batch ordering: Reproducible

Training data is ready.


In [40]:
import math
import torch

MONITOR_NONTOX_FACTOR = 1.50
MONITOR_BAD_CHECKS = 2
VALIDATION_SEED = SEED + 100

# Fixed validation subset for monitoring during training.
monitor_rows = random.Random(SEED + 701).sample(
    val_rows,
    min(MONITOR_VAL_PAIRS, len(val_rows))
)

assert {int(r["ames"]) for r in monitor_rows} == {0, 1}

model = model.to(DEVICE)


@torch.no_grad()
def evaluate_validation(rows, seed=VALIDATION_SEED):
    previous_rng = random.getstate()
    was_training = model.training

    totals = {
        "count": [0, 0],
        "ll": [0.0, 0.0],
        "ll_per_action": [0.0, 0.0],
        "weighted_loss": 0.0
    }

    model.eval()
    random.seed(seed)

    try:
        for start in range(0, len(rows), BATCH_SIZE):
            part = rows[start:start + BATCH_SIZE]
            batch = attach_proteins(part)

            ll = real_molecular_loglik(model, batch)
            labels = batch["tox_class"]

            if not torch.isfinite(ll).all():
                raise FloatingPointError(
                    "Invalid validation likelihood."
                )

            loss = finetuning_loss(
                ll, labels,
                FINETUNE_OBJECTIVE, TOX_LAMBDA
            )

            # INIT + bond-building actions + END.
            actions = batch["steps_per_molecule"] + 1
            ll_per_action = ll / actions

            totals["weighted_loss"] += loss.item() * len(part)

            for label in (0, 1):
                mask = labels == label
                count = int(mask.sum().item())

                if count:
                    totals["count"][label] += count
                    totals["ll"][label] += ll[mask].sum().item()
                    totals["ll_per_action"][label] += (
                        ll_per_action[mask].sum().item()
                    )
    finally:
        random.setstate(previous_rng)
        model.train(was_training)

    n0, n1 = totals["count"]
    assert n0 > 0 and n1 > 0

    result = {
        "pairs": n0 + n1,
        "non_toxic_pairs": n0,
        "toxic_pairs": n1,
        "non_toxic_nll": -totals["ll"][0] / n0,
        "non_toxic_nll_per_action": (
            -totals["ll_per_action"][0] / n0
        ),
        "toxic_ll": totals["ll"][1] / n1,
        "toxic_ll_per_action": (
            totals["ll_per_action"][1] / n1
        ),
        "weighted_loss": (
            totals["weighted_loss"] / (n0 + n1)
        )
    }

    assert all(math.isfinite(v) for v in result.values())
    return result


# Reference before fine-tuning begins.
initial_monitor = evaluate_validation(monitor_rows)

print("Initial validation reference")
print("----------------------------")
print(f"Validation examples: {initial_monitor['pairs']}")
print(
    "Non-toxic NLL: "
    f"{initial_monitor['non_toxic_nll']:.2f}"
)
print(
    "Toxic log-likelihood per action: "
    f"{initial_monitor['toxic_ll_per_action']:.2f}"
)
print("Validation outputs: Finite")

print("\nMonitoring is ready.")


Initial validation reference
----------------------------
Validation examples: 256
Non-toxic NLL: 62.17
Toxic log-likelihood per action: -1.67
Validation outputs: Finite

Monitoring is ready.


In [41]:
import os
import json
import time
import shutil

LAST_CKPT = RUN_DIR / "ckpt_last.pt"
BEST_CKPT = RUN_DIR / "ckpt_best.pt"
RUN_META = RUN_DIR / "run_meta.json"

TRAIN_LOG = RUN_DIR / "train_log.jsonl"
VAL_LOG = RUN_DIR / "val_log.jsonl"
MONITOR_LOG = RUN_DIR / "monitor_log.jsonl"

# Fixed experiment identity. Changes require a new run directory.
RUN_SIGNATURE = {
    "version": 1,
    "experiment": EXPERIMENT_NAME,
    "class": EXPERIMENT_CLASS,
    "pretraining": VARIANT,
    "finetuning_objective": FINETUNE_OBJECTIVE,
    "pretraining_steps": pretraining_steps,
    "pretraining_sha256": sha256_file(
        PRETRAIN_RUNS[VARIANT] / "checkpoint.pt"
    ),
    "dataset_manifest_sha256": sha256_file(
        DATA_DIR / "dataset_manifest.json"
    ),
    "train_sha256": manifest["output_sha256"]["train.tsv"],
    "val_sha256": manifest["output_sha256"]["val.tsv"],
    "atom_types_sha256": sha256_file(ATOM_TYPES_FILE),
    "model_config": config,
    "protein_size": N_C,
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "batching": "seeded_length_groups_all_pairs_v1",
    "k_paths": K_PATHS,
    "alpha": ALPHA,
    "tox_lambda": TOX_LAMBDA,
    "max_epochs": MAX_EPOCHS,
    "lr": LEARNING_RATE,
    "lr_decay": LR_DECAY,
    "lr_decay_step": LR_DECAY_STEP,
    "grad_clip": GRAD_CLIP,
    "patience": PATIENCE,
    "min_delta": MIN_DELTA,
    "monitor_every": MONITOR_EVERY,
    "monitor_pairs": len(monitor_rows),
    "monitor_factor": MONITOR_NONTOX_FACTOR,
    "monitor_bad_checks": MONITOR_BAD_CHECKS,
    "save_every": SAVE_EVERY,
    "log_every": LOG_EVERY,
    "train_pairs": len(train_rows),
    "val_pairs": len(val_rows),
    "numerical_policy": "log_probability_plus_1e-10"
}


In [42]:
def capture_rng():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available() else None
        )
    }


def restore_rng(saved):
    random.setstate(saved["python"])
    np.random.set_state(saved["numpy"])
    torch.set_rng_state(saved["torch"])

    if torch.cuda.is_available() and saved["cuda"] is not None:
        torch.cuda.set_rng_state_all(saved["cuda"])


def atomic_save(payload, path):
    temporary = path.with_name(path.name + f".tmp.{os.getpid()}")

    try:
        with temporary.open("wb") as file:
            torch.save(payload, file)
            file.flush()
            os.fsync(file.fileno())

        if path.exists():
            os.replace(
                path, path.with_name(path.name + ".prev")
            )

        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def write_log(path, entry):
    with path.open("a") as file:
        file.write(json.dumps(entry, allow_nan=False) + "\n")


def reconcile_logs(state):
    # Preserve and remove log entries newer than the checkpoint.
    for path in (TRAIN_LOG, VAL_LOG, MONITOR_LOG):
        if not path.exists():
            continue

        kept, orphaned = [], []

        for line in path.read_text().splitlines(keepends=True):
            try:
                entry = json.loads(line)

                newer = int(entry["step"]) > state["step"]

                if path == VAL_LOG:
                    newer |= (
                        int(entry["epoch"]) > state["epoch"]
                    )
            except (ValueError, KeyError, TypeError):
                newer = True

            (orphaned if newer else kept).append(line)

        if orphaned:
            backup = path.with_name(
                path.name + f".orphaned.{os.getpid()}"
            )
            shutil.copy2(path, backup)

            temporary = path.with_name(
                path.name + f".tmp.{os.getpid()}"
            )
            temporary.write_text("".join(kept))
            os.replace(temporary, path)

            print(f"Recovered previous log: {path.name}")


optimizer = None
train_state = None
monitor_reference = None


def save_checkpoint(reason="periodic", best=False):
    snapshot = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "state": dict(train_state),
        "rng": capture_rng(),
        "signature": RUN_SIGNATURE,
        "reason": reason
    }

    atomic_save(snapshot, LAST_CKPT)

    if best:
        atomic_save({
            "model": model.state_dict(),
            "signature": RUN_SIGNATURE,
            "step": train_state["step"],
            "epoch": train_state["epoch"],
            "validation_nll": train_state["best_nll"]
        }, BEST_CKPT)

In [43]:
def prepare_training_run():
    global optimizer, train_state, monitor_reference

    model.to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    train_state = {
        "epoch": 0,
        "next_batch": 0,
        "step": 0,
        "best_nll": float("inf"),
        "bad_epochs": 0,
        "monitor_bad": 0
    }

    existing = (
        RUN_DIR.exists() and any(RUN_DIR.iterdir())
    )

    if existing:
        if not RUN_META.exists():
            raise RuntimeError(
                "Existing run has no metadata. "
                "Use a different experiment directory."
            )

        saved_meta = json.loads(RUN_META.read_text())

        if saved_meta["signature"] != RUN_SIGNATURE:
            raise RuntimeError(
                "Experiment configuration has changed. "
                "Cannot safely resume."
            )

        monitor_reference = saved_meta["initial_monitor"]

        saved = None
        loaded_from = None

        for path in (
            LAST_CKPT,
            LAST_CKPT.with_name(LAST_CKPT.name + ".prev")
        ):
            if not path.exists():
                continue

            try:
                candidate = torch.load(
                    path,
                    map_location="cpu",
                    weights_only=False
                )
                if (
                    candidate["signature"] == RUN_SIGNATURE
                    and isinstance(candidate["state"], dict)
                ):
                    saved = candidate
                    loaded_from = path
                    break
            except Exception as exc:
                print(f"Checkpoint unreadable: {path.name}: {exc}")

        if saved is None:
            raise RuntimeError(
                "No compatible checkpoint found. "
                "Existing run has been preserved."
            )

        model.load_state_dict(saved["model"], strict=True)
        optimizer.load_state_dict(saved["optimizer"])

        # Restore Adam state tensors to their parameter devices.
        for parameter, values in optimizer.state.items():
            for key, value in values.items():
                if torch.is_tensor(value) and key != "step":
                    values[key] = value.to(parameter.device)

        train_state = saved["state"]

        assert set(train_state) == {
            "epoch", "next_batch", "step",
            "best_nll", "bad_epochs", "monitor_bad"
        }

        restore_rng(saved["rng"])
        reconcile_logs(train_state)

        print("Previous training run recovered.")
        print(f"Checkpoint: {loaded_from.name}")
        print(f"Completed steps: {train_state['step']:,}")
        print(f"Completed epochs: {train_state['epoch']}")

    else:
        RUN_DIR.mkdir(parents=True, exist_ok=True)

        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED)

        monitor_reference = initial_monitor

        metadata = {
            "signature": RUN_SIGNATURE,
            "initial_monitor": monitor_reference
        }

        temporary = RUN_META.with_name(
            RUN_META.name + f".tmp.{os.getpid()}"
        )
        temporary.write_text(json.dumps(metadata, indent=2))
        os.replace(temporary, RUN_META)

        # A checkpoint at step zero allows recovery even if
        # the first training batch fails.
        save_checkpoint(reason="initial")

        print("New training run initialized.")
        print("Initial checkpoint saved at step zero.")


print("Checkpoint and recovery functions are ready.")
print("Training has not started.")


Checkpoint and recovery functions are ready.
Training has not started.


In [44]:
import signal
import time

def check_training_progress():
    metrics = evaluate_validation(monitor_rows)

    reference_nll = monitor_reference["non_toxic_nll"]
    current_nll = metrics["non_toxic_nll"]

    # Stop only for substantial deterioration in non-toxic
    # likelihood, not for successful toxicity suppression.
    concerning = (
        current_nll > reference_nll * MONITOR_NONTOX_FACTOR
    )

    if concerning:
        train_state["monitor_bad"] += 1
    else:
        train_state["monitor_bad"] = 0

    record = {
        "step": train_state["step"],
        "epoch": train_state["epoch"],
        **metrics,
        "concerning": concerning,
        "consecutive_warnings": train_state["monitor_bad"]
    }

    write_log(MONITOR_LOG, record)

    print(
        f"Validation check | Step {train_state['step']:,} | "
        f"Non-toxic NLL {current_nll:.2f} | "
        f"Toxic LL/action {metrics['toxic_ll_per_action']:.2f}",
        flush=True
    )

    if concerning:
        print(
            f"Warning: Non-toxic NLL exceeds "
            f"{MONITOR_NONTOX_FACTOR:.1f}x its initial reference.",
            flush=True
        )

    if train_state["monitor_bad"] >= MONITOR_BAD_CHECKS:
        save_checkpoint(reason="validation_safety_stop")
        raise RuntimeError(
            "Training stopped after repeated non-toxic "
            "validation deterioration. Checkpoint saved."
        )


def run_finetuning():
    if not RUN_TRAINING:
        print("Training is disabled.")
        return

    assert VARIANT == "cdgcn_nll"
    assert FINETUNE_OBJECTIVE == "ctddg_eq8"
    assert not EVALUATE_TEST
    assert torch.cuda.is_available(), (
        "Full training requires a GPU allocation."
    )

    prepare_training_run()

    if (
        train_state["epoch"] >= MAX_EPOCHS
        or train_state["bad_epochs"] >= PATIENCE
    ):
        print("This experiment has already finished.")
        return

    stop_requested = {"value": False}
    previous_handlers = {}

    def request_stop(signum, frame):
        stop_requested["value"] = True
        print(
            "\nInterruption requested. Saving after the current batch.",
            flush=True
        )

    for sig in (signal.SIGINT, signal.SIGTERM):
        try:
            previous_handlers[sig] = signal.getsignal(sig)
            signal.signal(sig, request_stop)
        except (ValueError, OSError):
            pass

    started_at = time.monotonic()

    print("\nStarting BindingDB fine-tuning")
    print("------------------------------")
    print(f"Training pairs: {len(train_rows):,}")
    print(f"Batch size: {BATCH_SIZE}")
    print(f"Objective: CTDDG Eq. 8, lambda={TOX_LAMBDA}")
    print(f"Starting step: {train_state['step']:,}")
    print(f"Maximum epochs: {MAX_EPOCHS}\n", flush=True)

    try:
        while (
            train_state["epoch"] < MAX_EPOCHS
            and train_state["bad_epochs"] < PATIENCE
        ):
            epoch = train_state["epoch"]
            batches = make_training_batches(epoch)

            model.train()

            for batch_index in range(
                train_state["next_batch"], len(batches)
            ):
                if stop_requested["value"]:
                    save_checkpoint(reason="interrupted")
                    print("Training safely stopped and saved.")
                    return

                indices = batches[batch_index]
                rows = [train_rows[i] for i in indices]

                batch = attach_proteins(rows)

                optimizer.zero_grad(set_to_none=True)

                ll = real_molecular_loglik(model, batch)

                if not torch.isfinite(ll).all():
                    raise FloatingPointError(
                        "Invalid molecular likelihood. "
                        "Previous checkpoint preserved."
                    )

                loss = finetuning_loss(
                    ll,
                    batch["tox_class"],
                    FINETUNE_OBJECTIVE,
                    TOX_LAMBDA
                )

                if not torch.isfinite(loss):
                    raise FloatingPointError(
                        "Invalid training loss. "
                        "Previous checkpoint preserved."
                    )

                loss.backward()

                gradient_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    float("inf"),
                    error_if_nonfinite=True
                )

                if not torch.isfinite(gradient_norm):
                    raise FloatingPointError(
                        "Invalid gradients. "
                        "Previous checkpoint preserved."
                    )

                torch.nn.utils.clip_grad_value_(
                    model.parameters(), GRAD_CLIP
                )

                optimizer.step()

                train_state["step"] += 1
                train_state["next_batch"] = batch_index + 1

                step = train_state["step"]

                # Original multiplicative learning-rate decay.
                if step % LR_DECAY_STEP == 0:
                    for group in optimizer.param_groups:
                        group["lr"] *= (1.0 - LR_DECAY)

                if step == 1 or step % LOG_EVERY == 0:
                    with torch.no_grad():
                        labels = batch["tox_class"]

                        def class_ll(label):
                            selected = ll[labels == label]
                            return (
                                selected.mean().item()
                                if selected.numel() else None
                            )

                        non_ll = class_ll(0)
                        toxic_ll = class_ll(1)

                    record = {
                        "step": step,
                        "epoch": epoch + 1,
                        "batch": batch_index + 1,
                        "loss": loss.item(),
                        "ll_non_toxic": non_ll,
                        "ll_toxic": toxic_ll,
                        "gradient_norm": gradient_norm.item(),
                        "lr": optimizer.param_groups[0]["lr"],
                        "elapsed_seconds": round(
                            time.monotonic() - started_at, 1
                        )
                    }

                    write_log(TRAIN_LOG, record)

                    def show(value):
                        return (
                            f"{value:.2f}"
                            if value is not None else "N/A"
                        )

                    print(
                        f"Epoch {epoch + 1}/{MAX_EPOCHS} | "
                        f"Step {step:,} | "
                        f"Loss {loss.item():.2f} | "
                        f"Non-toxic LL {show(non_ll)} | "
                        f"Toxic LL {show(toxic_ll)} | "
                        f"Grad {gradient_norm.item():.1f}",
                        flush=True
                    )

                if step % MONITOR_EVERY == 0:
                    check_training_progress()

                if step % SAVE_EVERY == 0:
                    save_checkpoint(reason="periodic")

                if stop_requested["value"]:
                    save_checkpoint(reason="interrupted")
                    print("Training safely stopped and saved.")
                    return

            # Full validation at the end of each epoch.
            metrics = evaluate_validation(val_rows)
            current_nll = metrics["non_toxic_nll"]

            improved = (
                current_nll
                < train_state["best_nll"] - MIN_DELTA
            )

            if improved:
                train_state["best_nll"] = current_nll
                train_state["bad_epochs"] = 0
            else:
                train_state["bad_epochs"] += 1

            train_state["epoch"] = epoch + 1
            train_state["next_batch"] = 0

            write_log(VAL_LOG, {
                "step": train_state["step"],
                "epoch": train_state["epoch"],
                **metrics,
                "best_nll": train_state["best_nll"],
                "improved": improved,
                "bad_epochs": train_state["bad_epochs"]
            })

            save_checkpoint(
                reason="epoch_end",
                best=improved
            )

            print(
                f"\nEpoch {epoch + 1} complete | "
                f"Non-toxic val NLL {current_nll:.2f} | "
                f"Toxic val LL/action "
                f"{metrics['toxic_ll_per_action']:.2f} | "
                f"Best NLL {train_state['best_nll']:.2f}",
                flush=True
            )

            if train_state["bad_epochs"] >= PATIENCE:
                print("Early stopping: Validation stopped improving.")
                break

        print("\nFine-tuning finished.")
        print(f"Completed steps: {train_state['step']:,}")
        print(f"Completed epochs: {train_state['epoch']}")
        print(f"Best non-toxic validation NLL: {train_state['best_nll']:.2f}")
        print(f"Checkpoints saved in: {RUN_DIR}")

    finally:
        for sig, handler in previous_handlers.items():
            signal.signal(sig, handler)


print("Fine-tuning loop is ready.")
print("No optimizer steps have been executed.")


Fine-tuning loop is ready.
No optimizer steps have been executed.


In [45]:
assert VARIANT == "cdgcn_nll"
assert FINETUNE_OBJECTIVE == "ctddg_eq8"
assert TOX_LAMBDA == 0.1

assert EXPERIMENT_NAME == (
    "cdgcn_to_ctddg_d1_bs32_lambda0p1_v1"
)

assert RUN_SIGNATURE["tox_lambda"] == 0.1
assert RUN_SIGNATURE["experiment"] == EXPERIMENT_NAME

assert (RUN_DIR / "ckpt_last.pt").exists()

print("Correct experiment selected.")
print("Lambda: 0.1")
print("Existing checkpoint found.")
print("Ready to resume.")


Correct experiment selected.
Lambda: 0.1
Existing checkpoint found.
Ready to resume.


In [48]:
RUN_TRAINING = True

run_finetuning()

Recovered previous log: train_log.jsonl
Previous training run recovered.
Checkpoint: ckpt_last.pt
Completed steps: 2,400
Completed epochs: 0

Starting BindingDB fine-tuning
------------------------------
Training pairs: 118,765
Batch size: 32
Objective: CTDDG Eq. 8, lambda=0.1
Starting step: 2,400
Maximum epochs: 10

Epoch 1/10 | Step 2,450 | Loss 18.99 | Non-toxic LL -26.30 | Toxic LL -29.35 | Grad 15.7
Epoch 1/10 | Step 2,500 | Loss 15.50 | Non-toxic LL -26.53 | Toxic LL -28.77 | Grad 15.5
Epoch 1/10 | Step 2,550 | Loss 22.27 | Non-toxic LL -29.54 | Toxic LL -37.09 | Grad 18.7
Epoch 1/10 | Step 2,600 | Loss 15.01 | Non-toxic LL -24.44 | Toxic LL -30.08 | Grad 14.4
Validation check | Step 2,600 | Non-toxic NLL 42.17 | Toxic LL/action -1.16
Epoch 1/10 | Step 2,650 | Loss 17.81 | Non-toxic LL -28.84 | Toxic LL -32.44 | Grad 15.9
Epoch 1/10 | Step 2,700 | Loss 16.37 | Non-toxic LL -22.71 | Toxic LL -26.50 | Grad 13.9
Epoch 1/10 | Step 2,750 | Loss 13.65 | Non-toxic LL -26.34 | Toxic LL -